DNA26-1640: HLSA Report

In [0]:
%sql
create schema if not exists `p-bi`.hlsa

##### Create HLSA Members View

In [0]:
%sql
CREATE OR REPLACE VIEW `p-bi`.hlsa.members AS
WITH members AS (
    SELECT
        CurrentMonthLastWorkingDate AS BusinessDate,
        COUNT(DISTINCT MemberCode) AS CountOfMembers
    FROM `p-cdh`.primary.members m
    CROSS JOIN (
        SELECT DISTINCT
            CurrentMonthLastWorkingDate,
            PreviousMonthLastWorkingDate,
            FirstDateOfMonth
        FROM `p-cdh`.reference.ref_calendar
       -- WHERE CalendarDate BETWEEN '2024-01-01' AND '2026-05-31'
        WHERE CurrentMonthLastWorkingDate IN (
            SELECT DISTINCT CurrentMonthLastWorkingDate
            FROM `p-cdh`.reference.ref_calendar
            WHERE CurrentMonthLastWorkingDate <= (
                SELECT MAX(BusinessDate)
                FROM  `p-bi`.hlsa.exposures     
            )
            AND CurrentMonthLastWorkingDate >= (
                SELECT MIN(BusinessDate)
                FROM  `p-bi`.hlsa.exposures     
            )
            ORDER BY CurrentMonthLastWorkingDate DESC
        )
    )
    WHERE BusinessDate = (
        SELECT MAX(BusinessDate)
        FROM `p-cdh`.primary.members
    )
    AND LAST_DAY(MemberCreatedDate) = LAST_DAY(CurrentMonthLastWorkingDate)
    AND RegistrationStatus = 'VERIFIED'
    GROUP BY CurrentMonthLastWorkingDate
)
SELECT *
FROM members;

CREATE OR REPLACE VIEW `p-bi`.hlsa.members_bi_view AS
SELECT * 
FROM `p-bi`.hlsa.members 
WHERE BusinessDate IN (
SELECT DISTINCT BusinessDate 
FROM `p-bi`.hlsa.members 
ORDER BY BusinessDate DESC 
LIMIT 4
);

##### Create Complaints View

In [0]:
%sql
CREATE OR REPLACE VIEW `p-bi`.hlsa.complaints_bi_view AS
SELECT 
BusinessDate
,ComplaintID as Complaint_ID
,SentByCategory as Sent_by_category
,ComplaintCategory as Complaint_Category
,GeneralStatus as General_Status
,DateCreated as Date_Created
,DateCompleted as Date_Completed
,PipelineSoM
,NewInflowWithinMonthFlag
,CompletedWithinMonthFlag
,IngestionTimeStamp
FROM `p-bi`.hlsa.complaints 
WHERE BusinessDate IN (
SELECT DISTINCT BusinessDate 
FROM `p-bi`.hlsa.complaints 
ORDER BY BusinessDate DESC 
LIMIT 4
);

##### Create Requests View

In [0]:
%sql
CREATE OR REPLACE VIEW `p-bi`.hlsa.requests_bi_view AS
SELECT 
BusinessDate
,RequestID
,CommunicationChannel
,RequestCategory as Request_category
,GeneralStatus as General_Status
,DateCreated as Date_Created
,DateCompleted as Date_Completed
,PipelineSoM
,NewInflowWithinMonthFlag
,CompletedWithinMonthFlag
,Source
,IngestionTimestamp
FROM `p-bi`.hlsa.requests 
WHERE BusinessDate IN (
SELECT DISTINCT BusinessDate 
FROM `p-bi`.hlsa.requests 
ORDER BY BusinessDate DESC 
LIMIT 4
);

##### Create Exposure Report View

In [0]:
%sql
CREATE OR REPLACE VIEW `p-bi`.hlsa.exposures_bi_view AS
 
SELECT
    BusinessDate,
    Currency,
    dayspastdue,
    PortfolioName,
    ExposureKey,
    CDI,
    IFRSGrossCarryingAmount,
    GBV,
    outstandingtotalbalance,
    LoanType,
    LPSStatus,
    ResidentialCollateralFlag,
    IFRSGrossCarryingAmountEur,
    GBVEur,
    BogBalanceEur,
    outstandingtotalbalanceEur,
    ifnull(PartyType,'Individual') as PartyType,
    IsActiveFlag
FROM `p-bi`.hlsa.exposures
WHERE BusinessDate IN (
    SELECT DISTINCT CurrentMonthLastWorkingDate
    FROM `p-cdh`.reference.ref_calendar
    WHERE CurrentMonthLastWorkingDate <= (
        SELECT MAX(BusinessDate)
        FROM `p-bi`.hlsa.exposures
    )
    ORDER BY CurrentMonthLastWorkingDate DESC
    LIMIT 4
);

##### Create Collections Report View

In [0]:
%sql
CREATE OR REPLACE VIEW `p-bi`.hlsa.collections_bi_view AS
 
SELECT
    LoanType,
    PortfolioName,
    ActualDate,
    IFNULL(PaymentTypeGroup, 'Cash') PaymentTypeGroup,
    AmountEur,
    ExposureKey
FROM `p-bi`.hlsa.collections
WHERE (FinalReason = 'Matched' or PortfolioName like 'Core%')
    AND LoanType IS NOT NULL
    AND LAST_DAY(ActualDate) IN (
    SELECT DISTINCT LastDateOfMonth
    FROM `p-cdh`.reference.ref_calendar
    WHERE CurrentMonthLastWorkingDate <= (
        SELECT MAX(LAST_DAY(ActualDate))
        FROM `p-bi`.hlsa.collections
    )
    ORDER BY LastDateOfMonth DESC
    LIMIT 4
);

##### Create MED Modifications View

In [0]:
%sql
CREATE OR REPLACE VIEW `p-bi`.hlsa.medmodifications_bi_view AS

SELECT * 
FROM (
SELECT
    LoanType,
    BusinessDate,
    PortfolioGroup,
    COUNT(CASE WHEN ModifiedFlag NOT IN ('No-OCW','No-L3869','No-L4605','Non Modified') THEN ExposureKey END ) AS LoansNumber,
    COUNT(CASE WHEN ModifiedFlag NOT IN ('No-OCW','No-L3869','No-L4605','Non Modified') AND ResidentialCollateralFlag = 1 THEN ExposureKey END) AS LoansNumberWithCollateral,
    COUNT(DISTINCT CASE WHEN ModifiedFlag NOT IN ('No-OCW','No-L3869','No-L4605','Non Modified') THEN CDI END) AS BorrowerNumber,
    SUM(ModifiedCapital) AS TotalModifiedCapital,
    CASE WHEN (count(CASE WHEN AccountWoffFlag is not null THEN ExposureKey END)) > 0 THEN CAST(count(CASE WHEN AccountWoffFlag = 'ΔΑΝΕΙΟ ΜΕ ΔΙΑΓΡΑΦΗ ΟΦΕΙΛΗΣ' THEN ExposureKey END) AS FLOAT)
          / count(CASE WHEN AccountWoffFlag is not null THEN ExposureKey END) ELSE 0 END PercentageOfLoansWithWriteOff,
    1 - sum(WriteOffAmount) / sum(ModifiedCapital) WriteOffPercentage, TypeOfModification
  FROM
    `p-bi`.hlsa.medmodifications r
  GROUP BY
    LoanType,
    BusinessDate,
    PortfolioGroup,
    TypeOfModification

UNION ALL

SELECT
    'Total' AS LoanType,
    BusinessDate,
    PortfolioGroup,
    COUNT(CASE WHEN ModifiedFlag NOT IN ('No-OCW','No-L3869','No-L4605','Non Modified') THEN ExposureKey END ) AS LoansNumber,
    COUNT(CASE WHEN ModifiedFlag NOT IN ('No-OCW','No-L3869','No-L4605','Non Modified') AND ResidentialCollateralFlag = 1 THEN ExposureKey END) AS LoansNumberWithCollateral,
    COUNT(DISTINCT CASE WHEN ModifiedFlag NOT IN ('No-OCW','No-L3869','No-L4605','Non Modified') THEN CDI END) AS BorrowerNumber,
    SUM(ModifiedCapital) AS TotalModifiedCapital,
    CASE WHEN (count(CASE WHEN AccountWoffFlag is not null THEN ExposureKey END)) > 0 THEN CAST(count(CASE WHEN AccountWoffFlag = 'ΔΑΝΕΙΟ ΜΕ ΔΙΑΓΡΑΦΗ ΟΦΕΙΛΗΣ' THEN ExposureKey END) AS FLOAT)
          / count(CASE WHEN AccountWoffFlag is not null THEN ExposureKey END) ELSE 0 END PercentageOfLoansWithWriteOff,
    1 - sum(WriteOffAmount) / sum(ModifiedCapital) WriteOffPercentage, TypeOfModification
  FROM
    `p-bi`.hlsa.medmodifications r
  GROUP BY
    BusinessDate,
    PortfolioGroup,
    TypeOfModification) a
  WHERE BusinessDate IN (
    SELECT DISTINCT CurrentMonthLastWorkingDate
    FROM `p-cdh`.reference.ref_calendar
    WHERE CurrentMonthLastWorkingDate <= (
        SELECT MAX(BusinessDate)
        FROM `p-bi`.hlsa.exposures
    )
    ORDER BY CurrentMonthLastWorkingDate DESC
    LIMIT 4
  )

##### Create OCW 4738 View

In [0]:
%sql
CREATE OR REPLACE VIEW `p-bi`.hlsa.ocw4738 AS
 
WITH Products AS (
    SELECT DISTINCT
          ProductKey,
          ProductCode,
          Subproduct,
          ProductDescription
    FROM `p-cdh`.primary.product
    WHERE BusinessDate = (
        SELECT MAX(BusinessDate)
        FROM `p-cdh`.primary.product
    )
),
 
DatePerimeter AS (
    SELECT DISTINCT
        BusinessDate
    FROM `p-bi`.hlsa.exposures
    WHERE BusinessDate IN (
        SELECT DISTINCT CurrentMonthLastWorkingDate
        FROM `p-cdh`.reference.ref_calendar
        WHERE CurrentMonthLastWorkingDate <= (
            SELECT MAX(BusinessDate)
            FROM `p-bi`.hlsa.exposures
        )
        ORDER BY CurrentMonthLastWorkingDate DESC
    )
),
 
TaxPerimeter AS (
    SELECT DISTINCT
        L.PartyTaxRegistrationNumber,
        L.PortfolioName,
        L.OpenDate AS OCWAccountOpenDate,
        P.ProductDescription
    FROM `p-bi`.hlsa.exposures L
    INNER JOIN DatePerimeter d
        ON L.BusinessDate = d.BusinessDate
    LEFT JOIN Products P
        ON L.ProductKey = P.ProductKey
    WHERE P.ProductDescription LIKE '%4738%'
),
 
InitialLoans AS (
    SELECT
        L.ExposureKey,
        L.PartyTaxRegistrationNumber,
        L.PortfolioName,
        L.ProductKey,
        P.ProductCode,
        P.SubProduct,
        P.ProductDescription,
        MONTH(LAST_DAY(ADD_MONTHS(T.OCWAccountOpenDate, -1))) AS ReportingMonth,
        YEAR(LAST_DAY(ADD_MONTHS(T.OCWAccountOpenDate, -1))) AS ReportingYear
    FROM `p-bi`.hlsa.exposures L
    INNER JOIN DatePerimeter d
        ON L.BusinessDate = d.BusinessDate
    INNER JOIN TaxPerimeter T
        ON L.PartyTaxRegistrationNumber = T.PartyTaxRegistrationNumber
       AND L.PortfolioName = T.PortfolioName
    LEFT JOIN Products P
        ON L.ProductKey = P.ProductKey
    WHERE P.SubProduct NOT IN (
        '-',
        'PRF ΑΠΑΙΤΗΣΕΙΣ ΕΠΙΧΕΙΡΗΣΕΩΝ Μ.Λ.Τ',
        'PRF ΑΠΑΙΤΗΣΕΙΣ ΚΑΤΑΝΑΛΩΤΙΚΩΝ ΜΛΤ',
        'PRF ΜΕΡΙΚΗ ΑΠΟΣΒΕΣΗ ΟΦΕΙΛΗΣ',
        'PRF ΜΕΡΙΚΗ ΑΠΟΣΒΕΣΗ ΟΦΕΙΛΗΣ ΚΑΤΑΝΑΛΩΤΙΚΩΝ',
        'PRF ΜΕΡΙΚΗ ΑΠΟΣΒΕΣΗ ΟΦΕΙΛΗΣ ΣΤΕΓΑΣΤΙΚΩΝ',
        'PRF ΣΥΓΚ/ΣΗ ΑΠΟΣΒ ΟΦΕΙΛΩΝ ΚΑΤ/ΚΩΝ-ΧΩΡΙΣ HAIRCUT',
        'PRF ΣΥΓΚ/ΣΗ ΑΠΟΣΒ. ΟΦΕΙΛΩΝ ΚΑΤ/ΚΩΝ ΕΤΗΣΙΑ',
        'PRF ΣΥΓΚ/ΣΗ ΑΠΟΣΒ. ΟΦΕΙΛΩΝ ΚΑΤ/ΚΩΝ ΣΤΗ ΛΗΞΗ',
        'PRF ΣΥΓΚΕΝΤΡΩΣΗ ΑΠΟΣΒΕΣΜΕΝΩΝ ΟΦΕΙΛΩΝ',
        'PRF ΣΥΓΚΕΝΤΡΩΣΗ ΑΠΟΣΒΕΣΜΕΝΩΝ ΟΦΕΙΛΩΝ ΙΔ',
        'PRF ΤΟΚΟΙ Ν 3869 ΥΠΟ ΑΝΑΣΤΟΛΗ',
        'PRF ΤΟΚΟΙ Ν.3869 ΥΠΟ ΑΝΑΣΤΟΛΗ - ΚΑΤΑΝΑΛΩΤΙΚΑ',
        'PRF ΤΟΚΟΙ Ν.3869 ΥΠΟ ΑΝΑΣΤΟΛΗ- ΕΠΙΧΕΙΡΗΜΑΤΙΚΑ ΔΑΝΕΙΑ',
        'ΑΠΑΙΤΗΣΕΙΣ ΚΑΤΑΝΑΛΩΤΙΚΩΝ Μ.Λ.Τ. (Π/ΕΜΤΕ)',
        'ΑΠΑΙΤΗΣΕΙΣ ΜΛΤ',
        'ΜΕΡΙΚΗ ΑΠΟΣΒΕΣΗ ΟΦΕΙΛΗΣ ΕΠΙΧΕΙΡΗΜΑΤΙΚΩΝ',
        'ΜΕΡΙΚΗ ΑΠΟΣΒΕΣΗ ΟΦΕΙΛΗΣ ΚΑΤΑΝΑΛΩΤΙΚΩΝ',
        'ΥΠΟ ΑΙΡΕΣΗ ΔΙΑΓΡΑΦΗ - RET - ΡΥΘΜΙΣΕΙΣ(NEW_OCW)'
    )
    OR P.SubProduct IS NULL
),
 
BucketTable AS (
    SELECT
        L.PartyTaxRegistrationNumber,
        MAX(L.Bucket) AS FinalBucket,
        B.ReportingMonth,
        B.ReportingYear
    FROM `p-bi`.hlsa.exposures L
    INNER JOIN DatePerimeter d
        ON L.BusinessDate = d.BusinessDate
    INNER JOIN InitialLoans B
        ON L.ExposureKey = B.ExposureKey
       AND MONTH(L.BusinessDate) = B.ReportingMonth
       AND YEAR(L.BusinessDate) = B.ReportingYear
    WHERE LOWER(L.IsActiveFlag) = 'true'
    GROUP BY
        L.PartyTaxRegistrationNumber,
        B.ReportingMonth,
        B.ReportingYear
),
 
FinalPerimeter AS (
    SELECT
        L.BusinessDate,
        L.PortfolioName,
        L.PartyTaxRegistrationNumber,
        L.CDI,
        L.AgreementKey,
        L.outstandingtotalbalanceeur,
        L.OpenDate AS OCWAccountOpenDate,
        L.DaysPastDue,
        L.ExposureKey,
        L.Bucket
    FROM `p-bi`.hlsa.exposures L
    INNER JOIN DatePerimeter d
        ON L.BusinessDate = d.BusinessDate
    LEFT JOIN Products P
        ON L.ProductKey = P.ProductKey
    WHERE P.ProductDescription LIKE '%4738%'
      AND LOWER(L.IsActiveFlag) = 'true'
      AND L.outstandingtotalbalance > 0
),
 
Party AS (
    SELECT
        CDI,
        PartyType,
        BusinessDate
    FROM `p-cdh`.primary.party
    WHERE SourceSystem = 'profits'
),
 
FinalTable AS (
    SELECT DISTINCT
        f.BusinessDate,
        f.PortfolioName,
        f.ExposureKey,
        f.outstandingtotalbalanceeur,
        f.Bucket,
        b.ReportingMonth,
        b.ReportingYear,
        f.PartyTaxRegistrationNumber,
        CASE
            WHEN b.FinalBucket > 3 THEN 1
            ELSE 0
        END AS OldBucketGraterThan90DPD,
        p.PartyType,
        CASE
            WHEN MONTHS_BETWEEN(f.BusinessDate, f.OCWAccountOpenDate) > 6 THEN 1
            ELSE 0
        END AS OlderThan6MonthsFromModification
    FROM FinalPerimeter f
    LEFT JOIN BucketTable b
        ON f.PartyTaxRegistrationNumber = b.PartyTaxRegistrationNumber
       AND b.ReportingMonth = MONTH(f.BusinessDate)
       AND b.ReportingYear = YEAR(f.BusinessDate)
    LEFT JOIN Party p
        ON f.CDI = p.CDI
       AND f.BusinessDate = p.BusinessDate
    WHERE f.BusinessDate >= ADD_MONTHS(CURRENT_DATE(), -24)
)
 
SELECT
      BusinessDate
    , PortfolioName
    , COUNT(ExposureKey) AS ExposureKey
    , SUM(outstandingtotalbalanceeur) AS outstandingtotalbalanceeur
    , Bucket
    , ReportingMonth
    , ReportingYear
    , OldBucketGraterThan90DPD
    , PartyType
    , OlderThan6MonthsFromModification
FROM FinalTable
GROUP BY
      BusinessDate
    , PortfolioName
    , Bucket
    , ReportingMonth
    , ReportingYear
    , OldBucketGraterThan90DPD
    , PartyType
    , OlderThan6MonthsFromModification;

CREATE OR REPLACE VIEW `p-bi`.hlsa.ocw4738_bi_view AS
SELECT * 
FROM `p-bi`.hlsa.ocw4738
WHERE BusinessDate IN (
SELECT DISTINCT BusinessDate 
FROM `p-bi`.hlsa.ocw4738
ORDER BY BusinessDate DESC 
LIMIT 4
);

##### Create Acceptance Rates View

In [0]:
%sql
CREATE OR REPLACE VIEW `p-bi`.hlsa.acceptancerates AS
 
WITH Calendar AS (
 
    SELECT DISTINCT
        CurrentMonthLastWorkingDate as BusinessDate
    FROM `p-cdh`.reference.ref_calendar
    WHERE CurrentMonthLastWorkingDate <= (
        SELECT MAX(BusinessDate)
        FROM `p-bi`.hlsa.exposures
    )
    ORDER BY BusinessDate DESC),
 
OCWApplication AS (
 
    SELECT DISTINCT
          d.PortfolioName
        , d.CDI
        , d.TaxRegistrationNumber
        , d.ApplicationNumber
        , d.ApplicationStatus
        , d.ApplicationStatusDate
        , d.InitiationDate
        , d.SubmissionDate
        , d.CompletionReason
        , d.BusinessDate
        , CASE
              WHEN d.CompletionReason IN (
                    '104 - Εξωδικαστικός μηχανισμός - Σύμβαση αναδιάρθρωσης πολυμερούς',
                    '107 - Εξωδικαστικός μηχανισμός - Επιτυχία / Σύμβαση αναδιάρθρωσης διμερούς',
                    '110 - Επιτυχία / Σύμβαση αναδιάρθρωσης μεταξύ ΕΦΚΑ/Οφειλέτη',
                    '111 - Επιτυχία / Σύμβαση αναδιάρθρωσης μεταξύ ΑΑΔΕ/Οφειλέτη'
                )
              THEN 1 ELSE 0
          END AS CompletionReasonFlag
    FROM `p-cdh`.derivative.dunderlaw d
    INNER JOIN Calendar c
        ON c.BusinessDate = d.BusinessDate
    WHERE d.lawsystem = 'Εξωδικαστικός μηχανισμός'
      AND d.PortfolioName IS NOT NULL
 
),
 
CustomerBalance AS (
 
    SELECT
          BusinessDate
        , PartyTaxRegistrationNumber
        , SUM(outstandingtotalbalanceEur) AS outstandingtotalbalanceEur
    FROM `p-bi`.hlsa.exposures
    WHERE BusinessDate IN (
        SELECT DISTINCT CurrentMonthLastWorkingDate
        FROM `p-cdh`.reference.ref_calendar
    )
    GROUP BY
          BusinessDate
        , PartyTaxRegistrationNumber
 
),
 
FinalResults AS (
 
    SELECT
          O.*
        , CASE
              WHEN C.outstandingtotalbalanceEur <= 100000 THEN '100.000 €'
              WHEN C.outstandingtotalbalanceEur <= 150000 THEN '150.000 €'
              WHEN C.outstandingtotalbalanceEur <= 200000 THEN '200.000 €'
              WHEN C.outstandingtotalbalanceEur <= 250000 THEN '250.000 €'
              WHEN C.outstandingtotalbalanceEur <= 300000 THEN '300.000 €'
              WHEN C.outstandingtotalbalanceEur <= 350000 THEN '350.000 €'
              WHEN C.outstandingtotalbalanceEur <= 500000 THEN '500.000 €'
              WHEN C.outstandingtotalbalanceEur > 500000 THEN '500000+'
              ELSE 'N/A'
          END AS outstandingtotalbalanceEurGroup
 
        , CASE
              WHEN C.outstandingtotalbalanceEur <= 100000 THEN 1
              WHEN C.outstandingtotalbalanceEur <= 150000 THEN 2
              WHEN C.outstandingtotalbalanceEur <= 200000 THEN 3
              WHEN C.outstandingtotalbalanceEur <= 250000 THEN 4
              WHEN C.outstandingtotalbalanceEur <= 300000 THEN 5
              WHEN C.outstandingtotalbalanceEur <= 350000 THEN 6
              WHEN C.outstandingtotalbalanceEur <= 500000 THEN 7
              WHEN C.outstandingtotalbalanceEur > 500000 THEN 8
              ELSE 9
          END AS outstandingtotalbalanceEurGroupRank
 
    FROM OCWApplication O
    INNER JOIN CustomerBalance C
        ON O.TaxRegistrationNumber = C.PartyTaxRegistrationNumber
       AND O.BusinessDate = C.BusinessDate
 
)
 
SELECT
      outstandingtotalbalanceEurGroup
    , outstandingtotalbalanceEurGroupRank
    , BusinessDate
    , CompletionReasonFlag
    , COUNT(*) AS TotalApplications
    , COUNT(DISTINCT TaxRegistrationNumber) AS TaxRegistrationNumber
    , COUNT(DISTINCT ApplicationNumber) AS ApplicationNumber
 
FROM FinalResults
 
GROUP BY
      outstandingtotalbalanceEurGroup
    , outstandingtotalbalanceEurGroupRank
    , BusinessDate
    , CompletionReasonFlag
 
ORDER BY
      outstandingtotalbalanceEurGroupRank
    , BusinessDate;

CREATE OR REPLACE VIEW `p-bi`.hlsa.acceptancerates_bi_view AS
SELECT * 
FROM `p-bi`.hlsa.acceptancerates
WHERE BusinessDate IN (
SELECT DISTINCT BusinessDate 
FROM `p-bi`.hlsa.acceptancerates
ORDER BY BusinessDate DESC 
LIMIT 4
);

##### Create Dialer Incoming Calls View

In [0]:
%sql
CREATE OR REPLACE VIEW `p-bi`.hlsa.dialerincomingcalls AS
 
WITH Calendar AS (
    SELECT DISTINCT
        CurrentMonthLastWorkingDate,
        LastDateOfMonth
    FROM `p-cdh`.reference.ref_calendar
    --WHERE CalendarDate BETWEEN '2024-01-01' AND '2026-05-31'
    WHERE CurrentMonthLastWorkingDate <= (
        SELECT MAX(BusinessDate)
        FROM `p-bi`.hlsa.exposures
    )
    ORDER BY CurrentMonthLastWorkingDate DESC
),
 
AggregatedCalls AS (
    SELECT
        c.CurrentMonthLastWorkingDate AS BusinessDate,
 
        COUNT(DISTINCT CASE
            WHEN d.calltypecode = 'INBOUND'
              OR (
                    d.DialerType = 'qc'
                    AND d.CampaignName IN ('IN', 'ΙΝ', 'Default_Opencomm_IN')
                 )
            THEN d.dialerid
        END) AS num_of_incoming_calls,
 
        COUNT(DISTINCT CASE
            WHEN (
                    d.calltypecode = 'INBOUND'
                    AND LTRIM(RTRIM(IFNULL(d.talktimeduration, ''))) NOT IN ('', '0')
                 )
              OR (
                    d.DialerType = 'qc'
                    AND d.CampaignName IN ('IN', 'ΙΝ', 'Default_Opencomm_IN')
                 )
            THEN d.dialerid
        END) AS num_of_incoming_calls_answered,
 
        COUNT(DISTINCT CASE
            WHEN (
                    (
                        d.calltypecode = 'INBOUND'
                        AND LTRIM(RTRIM(IFNULL(d.talktimeduration, ''))) NOT IN ('', '0')
                    )
                    OR (
                        d.DialerType = 'qc'
                        AND d.CampaignName IN ('IN', 'ΙΝ', 'Default_Opencomm_IN')
                    )
                 )
             AND TRY_CAST(LTRIM(RTRIM(IFNULL(d.RingTimeDuration, ''))) AS INT) <= 60
            THEN d.dialerid
        END) AS num_of_incoming_calls_answered_within_60_seconds
 
    FROM `p-cdh`.primary.dialer d
 
    INNER JOIN Calendar c
        ON c.LastDateOfMonth = LAST_DAY(d.DateHour)
 
    WHERE d.IsActiveFlag = 'true'
    -- AND d.BusinessDate = (SELECT MAX(BusinessDate) FROM `p-cdh`.primary.dialer)
 
    GROUP BY
        c.CurrentMonthLastWorkingDate
)
 
SELECT
    BusinessDate,
    num_of_incoming_calls,
    num_of_incoming_calls_answered,
    num_of_incoming_calls_answered_within_60_seconds,
 
    ROUND(
        100.0 * (num_of_incoming_calls - num_of_incoming_calls_answered)
        / NULLIF(num_of_incoming_calls, 0),
        2
    ) AS abandoned_rate,
 
    ROUND(
        100.0 * num_of_incoming_calls_answered_within_60_seconds
        / NULLIF(num_of_incoming_calls, 0),
        2
    ) AS percentage_answered_within_60_seconds
 
FROM AggregatedCalls;

CREATE OR REPLACE VIEW `p-bi`.hlsa.dialerincomingcalls_bi_view AS
SELECT * 
FROM `p-bi`.hlsa.dialerincomingcalls
WHERE BusinessDate IN (
SELECT DISTINCT BusinessDate 
FROM `p-bi`.hlsa.dialerincomingcalls
ORDER BY BusinessDate DESC 
LIMIT 4
);

##### Create COR26 View

In [0]:
# %sql
# -- CREATE OR REPLACE VIEW `p-bi`.hlsa.cor26annex01_haps_bi_view AS
# CREATE OR REPLACE VIEW `p-bi`.hlsa.cor26annex01_haps AS

# select
#   cl.BusinessDate,
#   eba.Name EBAStatus,
#   ast.Name AssetSegment,
#   ifnull(sum(cor.LoanCount), 0) LoanCount,
#   cast(ifnull(sum(cor.Balance), 0) / 1000 as decimal(19, 2)) Balance,
#   cast(
#     ifnull(sum(cor.PendingAppLegalProtectionBalance), 0) / 1000 as decimal(19, 2)
#   ) PendingAppLegalProtectionBalance,
#   cast(ifnull(sum(cor.UnsecuredBalance), 0) / 1000 as decimal(19, 2)) UnsecuredBalance,
#   cast(ifnull(sum(cor.CollateralAmount), 0) / 1000 as decimal(19, 2)) CollateralAmount
# from
#   `p-cdh-delivery`.regulatoryreports.cordimassetsegment ast
#     cross join `p-cdh-delivery`.regulatoryreports.cordimebastatus eba
#     cross join (
#       select
#         max(CalendarDate) BusinessDate
#       from
#         `p-cdh`.reference.ref_calendar
#       where
#         IsBusinessDay = 1
#         and last_day(CalendarDate) = LastDateOfQuarter
#         and CalendarDate between '2025-01-01' and '2026-06-30'
#       group by
#         LastDateOfQuarter
#     ) cl
#     left join (
#       select
#         *,
#         rank() over (partition by BusinessDate order by RunVersion desc) as rnk
#       from
#         `p-cdh-delivery`.regulatoryreports.cor26annex01
#       where
#         PortfolioName in (
#           "Cosmos Retail",
#           "Cosmos Wholesale",
#           "Galaxy II Funding DAC",
#           "Galaxy IV Funding DAC",
#           "Orion X Securitisation DAC",
#           "RHODIUM PORTFOLIO - RETAIL",
#           "RHODIUM PORTFOLIO - WHOLESALE",
#           "MONZA PORTFOLIO - RETAIL",
#           "MONZA PORTFOLIO - WHOLESALE",
#           "DOMUS PORTFOLIO - RETAIL",
#           "DOMUS PORTFOLIO - WHOLESALE",
#           "GAIA II PORTFOLIO RETAIL",
#           "GAIA II PORTFOLIO WHOLESALE",
#           "GAIA I PORTFOLIO"
#         )
#     ) cor
#       on eba.Name = cor.EBAStatusCOR
#       and cor.AssetSegment = ast.Name
#       and cor.BusinessDate = cl.BusinessDate
#       and cor.RunFrequency = 'quarterly'
#       and cor.rnk = 1
# -- and cor.BusinessDate = '2025-12-31'
# -- and cor.PortfolioName in ( "Cosmos Retail", "Cosmos Wholesale","Galaxy II Funding DAC","Galaxy IV Funding DAC","Orion X Securitisation DAC" )
# group by
#   cl.BusinessDate,
#   eba.`Order`,
#   ast.`Order`,
#   eba.Name,
#   ast.Name
# -- order by
# --   cl.BusinessDate,
# --   ast.`Order`,
# --   eba.`Order`
# union
# select
#   cl.BusinessDate,
#   eba.Name EBAStatus,
#   ast.Name AssetSegment,
#   ifnull(sum(cor.LoanCount), 0) LoanCount,
#   cast(ifnull(sum(cor.Balance), 0) / 1000 as decimal(19, 2)) Balance,
#   cast(
#     ifnull(sum(cor.PendingAppLegalProtectionBalance), 0) / 1000 as decimal(19, 2)
#   ) PendingAppLegalProtectionBalance,
#   cast(ifnull(sum(cor.UnsecuredBalance), 0) / 1000 as decimal(19, 2)) UnsecuredBalance,
#   cast(ifnull(sum(cor.CollateralAmount), 0) / 1000 as decimal(19, 2)) CollateralAmount
# from
#   `p-cdh-delivery`.regulatoryreports.cordimassetsegment ast
#     cross join `p-cdh-delivery`.regulatoryreports.cordimebastatus eba
#     cross join (
#       select
#         max(CalendarDate) BusinessDate
#       from
#         `p-cdh`.reference.ref_calendar
#       where
#         IsBusinessDay = 1
#         and last_day(CalendarDate) = LastDateOfQuarter
#         -- and CalendarDate between '2024-01-01' and '2024-12-31'
#         and CalendarDate in (
#             SELECT DISTINCT
#                 CurrentMonthLastWorkingDate,
#                 LastDateOfMonth
#             FROM `p-cdh`.reference.ref_calendar
#             --WHERE CalendarDate BETWEEN '2024-01-01' AND '2026-05-31'
#             WHERE CurrentMonthLastWorkingDate <= (
#                   SELECT MAX(BusinessDate)
#                   FROM `p-bi`.hlsa.exposures
#             )
#             ORDER BY CurrentMonthLastWorkingDate DESC
#             LIMIT 4
#       )
#       group by
#         LastDateOfQuarter
#     ) cl
#     left join (
#       select
#         *,
#         rank() over (partition by ReferenceDate order by DatasetId desc) as rnk
#       from
#         `p-cdh-delivery`.reg.`_cor26annex01historical`
#       where
#         Portfolio in (
#           "Cosmos Retail",
#           "Cosmos Wholesale",
#           "Galaxy II Funding DAC",
#           "Galaxy IV Funding DAC",
#           "Orion X Securitisation DAC",
#           "RHODIUM PORTFOLIO - RETAIL",
#           "RHODIUM PORTFOLIO - WHOLESALE",
#           "MONZA PORTFOLIO - RETAIL",
#           "MONZA PORTFOLIO - WHOLESALE",
#           "DOMUS PORTFOLIO - RETAIL",
#           "DOMUS PORTFOLIO - WHOLESALE",
#           "GAIA II PORTFOLIO RETAIL",
#           "GAIA II PORTFOLIO WHOLESALE",
#           "GAIA I PORTFOLIO"
#         )
#     ) cor
#       on eba.Name = cor.EBAStatusCOR
#       and cor.AssetSegment = ast.Name
#       and cor.ReferenceDate = cl.BusinessDate
#       and cor.RunFrequency = 'quarterly'
#       and cor.rnk = 1
# -- and cor.BusinessDate = '2025-12-31'
# -- and cor.PortfolioName in ( "Cosmos Retail", "Cosmos Wholesale","Galaxy II Funding DAC","Galaxy IV Funding DAC","Orion X Securitisation DAC" )
# group by
#   cl.BusinessDate,
#   eba.`Order`,
#   ast.`Order`,
#   eba.Name,
#   ast.Name

In [0]:
# %sql
# -- CREATE OR REPLACE VIEW `p-bi`.hlsa.cor26annex01_nonhaps_bi_view AS
# CREATE OR REPLACE VIEW `p-bi`.common.cor26annex01_nonhaps AS

# select
#   cl.BusinessDate,
#   eba.Name EBAStatus,
#   ast.Name AssetSegment,
#   ifnull(sum(cor.LoanCount), 0) LoanCount,
#   cast(ifnull(sum(cor.Balance), 0) / 1000 as decimal(19, 2)) Balance,
#   cast(
#     ifnull(sum(cor.PendingAppLegalProtectionBalance), 0) / 1000 as decimal(19, 2)
#   ) PendingAppLegalProtectionBalance,
#   cast(ifnull(sum(cor.UnsecuredBalance), 0) / 1000 as decimal(19, 2)) UnsecuredBalance,
#   cast(ifnull(sum(cor.CollateralAmount), 0) / 1000 as decimal(19, 2)) CollateralAmount
# from
#   `p-cdh-delivery`.regulatoryreports.cordimassetsegment ast
#     cross join `p-cdh-delivery`.regulatoryreports.cordimebastatus eba
#     cross join (
#       select
#         max(CalendarDate) BusinessDate
#       from
#         `p-cdh`.reference.ref_calendar
#       where
#         IsBusinessDay = 1
#         and last_day(CalendarDate) = LastDateOfQuarter
#         and CalendarDate between '2025-01-01' and '2026-06-30'
#       group by
#         LastDateOfQuarter
#     ) cl
#     left join (
#       select
#         *,
#         rank() over (partition by BusinessDate order by RunVersion desc) as rnk
#       from
#         `p-cdh-delivery`.regulatoryreports.cor26annex01
#       where
#         PortfolioName not in (
#           "Cosmos Retail",
#           "Cosmos Wholesale",
#           "Galaxy II Funding DAC",
#           "Galaxy IV Funding DAC",
#           "Orion X Securitisation DAC",
#           "RHODIUM PORTFOLIO - RETAIL",
#           "RHODIUM PORTFOLIO - WHOLESALE",
#           "MONZA PORTFOLIO - RETAIL",
#           "MONZA PORTFOLIO - WHOLESALE",
#           "DOMUS PORTFOLIO - RETAIL",
#           "DOMUS PORTFOLIO - WHOLESALE",
#           "GAIA II PORTFOLIO RETAIL",
#           "GAIA II PORTFOLIO WHOLESALE",
#           "GAIA I PORTFOLIO"
#         )
#     ) cor
#       on eba.Name = cor.EBAStatusCOR
#       and cor.AssetSegment = ast.Name
#       and cor.BusinessDate = cl.BusinessDate
#       and cor.RunFrequency = 'quarterly'
#       and cor.rnk = 1
# -- and cor.BusinessDate = '2025-12-31'
# -- and cor.PortfolioName in ( "Cosmos Retail", "Cosmos Wholesale","Galaxy II Funding DAC","Galaxy IV Funding DAC","Orion X Securitisation DAC" )
# group by
#   cl.BusinessDate,
#   eba.`Order`,
#   ast.`Order`,
#   eba.Name,
#   ast.Name
# -- order by
# --   cl.BusinessDate,
# --   ast.`Order`,
# --   eba.`Order`
# union
# select
#   cl.BusinessDate,
#   eba.Name EBAStatus,
#   ast.Name AssetSegment,
#   ifnull(sum(cor.LoanCount), 0) LoanCount,
#   cast(ifnull(sum(cor.Balance), 0) / 1000 as decimal(19, 2)) Balance,
#   cast(
#     ifnull(sum(cor.PendingAppLegalProtectionBalance), 0) / 1000 as decimal(19, 2)
#   ) PendingAppLegalProtectionBalance,
#   cast(ifnull(sum(cor.UnsecuredBalance), 0) / 1000 as decimal(19, 2)) UnsecuredBalance,
#   cast(ifnull(sum(cor.CollateralAmount), 0) / 1000 as decimal(19, 2)) CollateralAmount
# from
#   `p-cdh-delivery`.regulatoryreports.cordimassetsegment ast
#     cross join `p-cdh-delivery`.regulatoryreports.cordimebastatus eba
#     cross join (
#       select
#         max(CalendarDate) BusinessDate
#       from
#         `p-cdh`.reference.ref_calendar
#       where
#         IsBusinessDay = 1
#         and last_day(CalendarDate) = LastDateOfQuarter
#         and CalendarDate between '2024-01-01' and '2024-12-31'
#       group by
#         LastDateOfQuarter
#     ) cl
#     left join (
#       select
#         *,
#         rank() over (partition by ReferenceDate order by DatasetId desc) as rnk
#       from
#         `p-cdh-delivery`.reg.`_cor26annex01historical`
#       where
#         Portfolio not in (
#           "Cosmos Retail",
#           "Cosmos Wholesale",
#           "Galaxy II Funding DAC",
#           "Galaxy IV Funding DAC",
#           "Orion X Securitisation DAC",
#           "RHODIUM PORTFOLIO - RETAIL",
#           "RHODIUM PORTFOLIO - WHOLESALE",
#           "MONZA PORTFOLIO - RETAIL",
#           "MONZA PORTFOLIO - WHOLESALE",
#           "DOMUS PORTFOLIO - RETAIL",
#           "DOMUS PORTFOLIO - WHOLESALE",
#           "GAIA II PORTFOLIO RETAIL",
#           "GAIA II PORTFOLIO WHOLESALE",
#           "GAIA I PORTFOLIO"
#         )
#     ) cor
#       on eba.Name = cor.EBAStatusCOR
#       and cor.AssetSegment = ast.Name
#       and cor.ReferenceDate = cl.BusinessDate
#       and cor.RunFrequency = 'quarterly'
#       and cor.rnk = 1
# -- and cor.BusinessDate = '2025-12-31'
# -- and cor.PortfolioName in ( "Cosmos Retail", "Cosmos Wholesale","Galaxy II Funding DAC","Galaxy IV Funding DAC","Orion X Securitisation DAC" )
# group by
#   cl.BusinessDate,
#   eba.`Order`,
#   ast.`Order`,
#   eba.Name,
#   ast.Name

In [0]:
%sql
CREATE OR REPLACE VIEW `p-bi`.hlsa.cor26annex01 AS

WITH Calendar AS (
    SELECT DISTINCT
        CurrentMonthLastWorkingDate,
        LastDateOfMonth
    FROM `p-cdh`.reference.ref_calendar
    --WHERE CalendarDate BETWEEN '2024-01-01' AND '2026-05-31'
    WHERE CurrentMonthLastWorkingDate <= (
        SELECT MAX(BusinessDate)
        FROM `p-bi`.hlsa.exposures
    )
    ORDER BY CurrentMonthLastWorkingDate DESC
),
cor26annex01 (
select
  cl.CurrentMonthLastWorkingDate BusinessDate,
  eba.Name EBAStatus,
  ast.Name AssetSegment,
  ifnull(sum(cor.LoanCount), 0) LoanCount,
  cast(ifnull(sum(cor.Balance), 0) / 1000 as decimal(19, 2)) Balance,
  cast(
    ifnull(sum(cor.PendingAppLegalProtectionBalance), 0) / 1000 as decimal(19, 2)
  ) PendingAppLegalProtectionBalance,
  cast(ifnull(sum(cor.UnsecuredBalance), 0) / 1000 as decimal(19, 2)) UnsecuredBalance,
  cast(ifnull(sum(cor.CollateralAmount), 0) / 1000 as decimal(19, 2)) CollateralAmount,
  'Haps' PortfolioGroup
from
  `p-cdh-delivery`.regulatoryreports.cordimassetsegment ast
    cross join `p-cdh-delivery`.regulatoryreports.cordimebastatus eba
    cross join Calendar cl
    left join (
      select
        *,
        rank() over (partition by BusinessDate order by RunVersion desc) as rnk
      from
        `p-cdh-delivery`.regulatoryreports.cor26annex01
      where
        PortfolioName in (
          "Cosmos Retail",
          "Cosmos Wholesale",
          "Galaxy II Funding DAC",
          "Galaxy IV Funding DAC",
          "Orion X Securitisation DAC",
          "RHODIUM PORTFOLIO - RETAIL",
          "RHODIUM PORTFOLIO - WHOLESALE",
          "MONZA PORTFOLIO - RETAIL",
          "MONZA PORTFOLIO - WHOLESALE",
          "DOMUS PORTFOLIO - RETAIL",
          "DOMUS PORTFOLIO - WHOLESALE",
          "GAIA II PORTFOLIO RETAIL",
          "GAIA II PORTFOLIO WHOLESALE",
          "GAIA I PORTFOLIO"
        )
    ) cor
      on eba.Name = cor.EBAStatusCOR
      and cor.AssetSegment = ast.Name
      and cor.BusinessDate = cl.CurrentMonthLastWorkingDate
      and cor.RunFrequency = 'quarterly'
      and cor.rnk = 1
-- and cor.BusinessDate = '2025-12-31'
-- and cor.PortfolioName in ( "Cosmos Retail", "Cosmos Wholesale","Galaxy II Funding DAC","Galaxy IV Funding DAC","Orion X Securitisation DAC" )
group by
  cl.CurrentMonthLastWorkingDate,
  eba.`Order`,
  ast.`Order`,
  eba.Name,
  ast.Name
-- order by
--   cl.BusinessDate,
--   ast.`Order`,
--   eba.`Order`
union all
select
  cl.CurrentMonthLastWorkingDate BusinessDate,
  eba.Name EBAStatus,
  ast.Name AssetSegment,
  ifnull(sum(cor.LoanCount), 0) LoanCount,
  cast(ifnull(sum(cor.Balance), 0) / 1000 as decimal(19, 2)) Balance,
  cast(
    ifnull(sum(cor.PendingAppLegalProtectionBalance), 0) / 1000 as decimal(19, 2)
  ) PendingAppLegalProtectionBalance,
  cast(ifnull(sum(cor.UnsecuredBalance), 0) / 1000 as decimal(19, 2)) UnsecuredBalance,
  cast(ifnull(sum(cor.CollateralAmount), 0) / 1000 as decimal(19, 2)) CollateralAmount,
  'Haps' PortfolioGroup
from
  `p-cdh-delivery`.regulatoryreports.cordimassetsegment ast
    cross join `p-cdh-delivery`.regulatoryreports.cordimebastatus eba
    cross join Calendar cl
    left join (
      select
        *,
        rank() over (partition by ReferenceDate order by DatasetId desc) as rnk
      from
        `p-cdh-delivery`.reg.`_cor26annex01historical`
      where
        Portfolio in (
          "Cosmos Retail",
          "Cosmos Wholesale",
          "Galaxy II Funding DAC",
          "Galaxy IV Funding DAC",
          "Orion X Securitisation DAC",
          "RHODIUM PORTFOLIO - RETAIL",
          "RHODIUM PORTFOLIO - WHOLESALE",
          "MONZA PORTFOLIO - RETAIL",
          "MONZA PORTFOLIO - WHOLESALE",
          "DOMUS PORTFOLIO - RETAIL",
          "DOMUS PORTFOLIO - WHOLESALE",
          "GAIA II PORTFOLIO RETAIL",
          "GAIA II PORTFOLIO WHOLESALE",
          "GAIA I PORTFOLIO"
        )
    ) cor
      on eba.Name = cor.EBAStatusCOR
      and cor.AssetSegment = ast.Name
      and cor.ReferenceDate = cl.CurrentMonthLastWorkingDate
      and cor.RunFrequency = 'quarterly'
      and cor.rnk = 1
-- and cor.BusinessDate = '2025-12-31'
-- and cor.PortfolioName in ( "Cosmos Retail", "Cosmos Wholesale","Galaxy II Funding DAC","Galaxy IV Funding DAC","Orion X Securitisation DAC" )
group by
  cl.CurrentMonthLastWorkingDate,
  eba.`Order`,
  ast.`Order`,
  eba.Name,
  ast.Name

union all

select
  cl.CurrentMonthLastWorkingDate BusinessDate,
  eba.Name EBAStatus,
  ast.Name AssetSegment,
  ifnull(sum(cor.LoanCount), 0) LoanCount,
  cast(ifnull(sum(cor.Balance), 0) / 1000 as decimal(19, 2)) Balance,
  cast(
    ifnull(sum(cor.PendingAppLegalProtectionBalance), 0) / 1000 as decimal(19, 2)
  ) PendingAppLegalProtectionBalance,
  cast(ifnull(sum(cor.UnsecuredBalance), 0) / 1000 as decimal(19, 2)) UnsecuredBalance,
  cast(ifnull(sum(cor.CollateralAmount), 0) / 1000 as decimal(19, 2)) CollateralAmount,
  'Non-Haps' as PortfolioGroup
from
  `p-cdh-delivery`.regulatoryreports.cordimassetsegment ast
    cross join `p-cdh-delivery`.regulatoryreports.cordimebastatus eba
    cross join Calendar cl
    left join (
      select
        *,
        rank() over (partition by BusinessDate order by RunVersion desc) as rnk
      from
        `p-cdh-delivery`.regulatoryreports.cor26annex01
      where
        PortfolioName not in (
          "Cosmos Retail",
          "Cosmos Wholesale",
          "Galaxy II Funding DAC",
          "Galaxy IV Funding DAC",
          "Orion X Securitisation DAC",
          "RHODIUM PORTFOLIO - RETAIL",
          "RHODIUM PORTFOLIO - WHOLESALE",
          "MONZA PORTFOLIO - RETAIL",
          "MONZA PORTFOLIO - WHOLESALE",
          "DOMUS PORTFOLIO - RETAIL",
          "DOMUS PORTFOLIO - WHOLESALE",
          "GAIA II PORTFOLIO RETAIL",
          "GAIA II PORTFOLIO WHOLESALE",
          "GAIA I PORTFOLIO"
        )
    ) cor
      on eba.Name = cor.EBAStatusCOR
      and cor.AssetSegment = ast.Name
      and cor.BusinessDate = cl.CurrentMonthLastWorkingDate
      and cor.RunFrequency = 'quarterly'
      and cor.rnk = 1
-- and cor.BusinessDate = '2025-12-31'
-- and cor.PortfolioName in ( "Cosmos Retail", "Cosmos Wholesale","Galaxy II Funding DAC","Galaxy IV Funding DAC","Orion X Securitisation DAC" )
group by
  cl.CurrentMonthLastWorkingDate,
  eba.`Order`,
  ast.`Order`,
  eba.Name,
  ast.Name
-- order by
--   cl.BusinessDate,
--   ast.`Order`,
--   eba.`Order`
union all
select
  cl.CurrentMonthLastWorkingDate as BusinessDate,
  eba.Name EBAStatus,
  ast.Name AssetSegment,
  ifnull(sum(cor.LoanCount), 0) LoanCount,
  cast(ifnull(sum(cor.Balance), 0) / 1000 as decimal(19, 2)) Balance,
  cast(
    ifnull(sum(cor.PendingAppLegalProtectionBalance), 0) / 1000 as decimal(19, 2)
  ) PendingAppLegalProtectionBalance,
  cast(ifnull(sum(cor.UnsecuredBalance), 0) / 1000 as decimal(19, 2)) UnsecuredBalance,
  cast(ifnull(sum(cor.CollateralAmount), 0) / 1000 as decimal(19, 2)) CollateralAmount,
  'Non-Haps' as PortfolioGroup
from
  `p-cdh-delivery`.regulatoryreports.cordimassetsegment ast
    cross join `p-cdh-delivery`.regulatoryreports.cordimebastatus eba
    cross join Calendar cl
    left join (
      select
        *,
        rank() over (partition by ReferenceDate order by DatasetId desc) as rnk
      from
        `p-cdh-delivery`.reg.`_cor26annex01historical`
      where
        Portfolio not in (
          "Cosmos Retail",
          "Cosmos Wholesale",
          "Galaxy II Funding DAC",
          "Galaxy IV Funding DAC",
          "Orion X Securitisation DAC",
          "RHODIUM PORTFOLIO - RETAIL",
          "RHODIUM PORTFOLIO - WHOLESALE",
          "MONZA PORTFOLIO - RETAIL",
          "MONZA PORTFOLIO - WHOLESALE",
          "DOMUS PORTFOLIO - RETAIL",
          "DOMUS PORTFOLIO - WHOLESALE",
          "GAIA II PORTFOLIO RETAIL",
          "GAIA II PORTFOLIO WHOLESALE",
          "GAIA I PORTFOLIO"
        )
    ) cor
      on eba.Name = cor.EBAStatusCOR
      and cor.AssetSegment = ast.Name
      and cor.ReferenceDate = cl.CurrentMonthLastWorkingDate
      and cor.RunFrequency = 'quarterly'
      and cor.rnk = 1
-- and cor.BusinessDate = '2025-12-31'
-- and cor.PortfolioName in ( "Cosmos Retail", "Cosmos Wholesale","Galaxy II Funding DAC","Galaxy IV Funding DAC","Orion X Securitisation DAC" )
group by
  cl.CurrentMonthLastWorkingDate,
  eba.`Order`,
  ast.`Order`,
  eba.Name,
  ast.Name
  )
  select * from cor26annex01;

CREATE OR REPLACE VIEW `p-bi`.hlsa.cor26annex01_bi_view AS
SELECT * 
FROM `p-bi`.hlsa.cor26annex01
WHERE BusinessDate IN (
SELECT DISTINCT BusinessDate 
FROM `p-bi`.hlsa.cor26annex01
ORDER BY BusinessDate DESC 
LIMIT 4
);

In [0]:
%sql
CREATE OR REPLACE VIEW `p-bi`.hlsa.dim_communication_channel AS

SELECT 'fotdelli' AS UserName, 'CEPAL' AS CreatedByChannel, 'Customer Service E - Mail' AS CommunicationChannel
UNION ALL SELECT 'marsenko', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'mtampa', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'markarapiperi', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'lmicha', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'gtzalakostas', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'stsemperopoulos', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'empouzala', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'czygouri', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'malibate', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'vkaoura', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'Cnikolakopoulos', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'psavvopoulos', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'agkolfinopoulou', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'vpanagopoulos', 'CEPAL', 'Τηλεφωνική εξυπηρέτηση'
UNION ALL SELECT 'melvmikropoulou', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'okariakli', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'mchatzisaroglou', 'CEPAL', 'Τηλεφωνική εξυπηρέτηση'
UNION ALL SELECT 'mmavropoulou', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'msotiriadou', 'CEPAL', 'Τηλεφωνική εξυπηρέτηση'
UNION ALL SELECT 'ckolofotias', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'stsantila', 'CEPAL', 'Τηλεφωνική εξυπηρέτηση'
UNION ALL SELECT 'angvassalou', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'marzisimopoulou', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'stapetsi', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'extepanatazopoulou', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'stychaloulakou', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'epapagianni', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'sioufkzervini', 'Sioufas', 'Λοιπά'
UNION ALL SELECT 'eseitanidi', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'atsiouri', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'extdagrimaki', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'cgialitakis', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'eiordanidou', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'esalagianni', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'emanoli', 'CEPAL', 'Τηλεφωνική εξυπηρέτηση'
UNION ALL SELECT 'nanagnostopoulos', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'aletasi', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'sthomakos', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'mkoviou', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'fmoraiti', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'garmoyze', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'cantoniou', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'gpapachristou', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'mpanteli', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'ioagkrinia', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'etsouloufi', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'atsekoura', 'CEPAL', 'Τηλεφωνική εξυπηρέτηση'
UNION ALL SELECT 'astathakopoulou', 'CEPAL', 'Τηλεφωνική εξυπηρέτηση'
UNION ALL SELECT 'mkremyda', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'elegrammenou', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'skouliouras', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'konxystras', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'amaritsa', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'psyvkritsotaki', 'Psychas', 'Λοιπά'
UNION ALL SELECT 'aanasoglou', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'eirplatana', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'antkeramidas', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'eanastasiadi', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'konkiousis', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'mvlachou', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'extageorgiou', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'extapagona', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'extvpatsos', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'vgkotzi', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'extevika', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'extstasoulas', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'mkoutsogianni', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'extekanaki', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'nikstathopoulou', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'extcchronopoulou', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'extacharmpas', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'extegeorgousi', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'extamakridou', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'dchrysou', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'extpbouranakou', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'ekokoli', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'extpalexopoulou', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'extnkladis', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'ekonstantinidi', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'kspyrou', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'geovrachas', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'extkkolicheva', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'extvalexopoulou', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'mkeskou', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'epontikakis', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'charegkavou', 'Charaktiniotis', 'Λοιπά'
UNION ALL SELECT 'konfortotira', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'mkanaki', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'omicha', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'gantoniou', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'charceleftheriou', 'Charaktiniotis', 'Λοιπά'
UNION ALL SELECT 'extmkalousa', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'palevizou', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'extamakraki', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'elenimorakiotaki', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'extmyfantopoulou', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'skyrkos', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'nkatsimpiri', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'emathioudaki', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'zafrappou', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'nikkapnisis', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'chrnastopoulos', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'sioufstsonoglou', 'Sioufas', 'Λοιπά'
UNION ALL SELECT 'stzanis', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'gsiskou', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'dsinani', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'mkoulouri', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'gravmefremidou', 'Marakakis', 'Λοιπά'
UNION ALL SELECT 'theladas', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'extcvelona', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'okakisi', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'extvvelikadi', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'extlmylona', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'vsinou', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'extogkioka', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'epapagrigoriou', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'extpmigkou', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'gpaliouras', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'afrgeorgikou', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'masvou', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'skontou', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'dimlampropoulou', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'ekatsigaraki', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'mralli', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'cmperedimas', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'stavmrizoglou', 'Stavropoulou', 'Λοιπά'
UNION ALL SELECT 'dgiatroudaki', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'extitsochalis', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'esmargianakis', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'vrigopoulos', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'extdkoutonia', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'extapatta', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'tsafdtsakiri', 'Tsafaras', 'Λοιπά'
UNION ALL SELECT 'extmkarampatsou', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'nathanasoulas', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'mprekatsounaki', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'dimpapadakis', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'extapapadopoulou', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'extskiosi', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'extepapaioannou', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'extagkolovko', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'mmarkou', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'vkardara', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'kkatsa', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'emegge', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'extmnikolopoulou', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'extpmoutopoulou', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'extppapathanasiou', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'mpsimadi', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'extfgennata', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'ktsaggou', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'extpsavvopoulos', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'extcpanagodimitropou', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'extechatzikosta', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'epapadopoulou', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'nikviglas', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'pkoutromanos', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'extgpotiris', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'extmptochou', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'kchiraki', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'geologariastaki', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'extefotiou', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'ptriantafylli', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'extigalanopoulou', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'dchatzakis', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'extcnikolakou', 'Psychas', 'Λοιπά'
UNION ALL SELECT 'aribougioukloglou', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'extmtzikopoulou', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'kkaradima', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'extkbeka', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'astavrianoudaki', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'extanikolaou', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'extdtsantis', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'extamakris', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'extamakrogamvrakis', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'egianniou', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'extaplatias', 'Sioufas', 'Λοιπά'
UNION ALL SELECT 'charezagarelou', 'Charaktiniotis', 'Λοιπά'
UNION ALL SELECT 'fstamatelatou', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'ioaangelopoulou', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'extgtselika', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'nplspot', 'SSPUnit', 'Portal'
UNION ALL SELECT 'fstamatelatou@cepal.gr', 'DRMU', 'Λοιπά'
UNION ALL SELECT 'ioanna.gkrinia@cepal.gr', 'DRMU', 'Λοιπά'
UNION ALL SELECT 'mariastylia.senko@cepal.gr', 'DRMU', 'Customer Service E - Mail'
UNION ALL SELECT 'eirini.platana@cepal.gr', 'DRMU', 'Λοιπά'
UNION ALL SELECT 'gkotseras@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'cmperedimas@cepal.gr', 'FrontlineRetail', 'Λοιπά'
UNION ALL SELECT 'czygouri@cepal.gr', 'DRMU', 'Customer Service E - Mail'
UNION ALL SELECT 'eseitanidi@cepal.gr', 'FrontlineRetail', 'Λοιπά'
UNION ALL SELECT 'msotiriadou@cepal.gr', 'DRMU', 'Τηλεφωνική εξυπηρέτηση'
UNION ALL SELECT 'christos.charalampopoulos@cepal.gr', 'FrontlineRetail', 'Λοιπά'
UNION ALL SELECT 'gsallaris@cepal.gr', 'FrontlineRetail', 'Λοιπά'
UNION ALL SELECT 'maria.karapiperi@cepal.gr', 'DRMU', 'Customer Service E - Mail'
UNION ALL SELECT 'agkolfinopoulou@cepal.gr', 'DRMU', 'Customer Service E - Mail'
UNION ALL SELECT 'vsinou@cepal.gr', 'FrontlineRetail', 'Λοιπά'
UNION ALL SELECT 'fmoraiti@cepal.gr', 'FrontlineRetail', 'Λοιπά'
UNION ALL SELECT 'stsemperopoulos@cepal.gr', 'DRMU', 'Customer Service E - Mail'
UNION ALL SELECT 'mchatzisaroglou@cepal.gr', 'DRMU', 'Τηλεφωνική εξυπηρέτηση'
UNION ALL SELECT 'lpolychronidou@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'eiordanidou@cepal.gr', 'DRMU', 'Λοιπά'
UNION ALL SELECT 'lmicha@cepal.gr', 'DRMU', 'Customer Service E - Mail'
UNION ALL SELECT 'xgatou@cepal.gr', 'FrontlineRetail', 'Λοιπά'
UNION ALL SELECT 'cgialitakis@cepal.gr', 'FrontlineRetail', 'Λοιπά'
UNION ALL SELECT 'mkanaki@cepal.gr', 'FrontlineRetail', 'Λοιπά'
UNION ALL SELECT 'atsekoura@cepal.gr', 'DRMU', 'Τηλεφωνική εξυπηρέτηση'
UNION ALL SELECT 'eirini.kompinou@cepal.gr', 'FrontlineRetail', 'Λοιπά'
UNION ALL SELECT 'malibate@cepal.gr', 'DRMU', 'Customer Service E - Mail'
UNION ALL SELECT 'gkolgioni@cepal.gr', 'FrontlineRetail', 'Λοιπά'
UNION ALL SELECT 'epapagrigoriou@cepal.gr', 'FrontlineRetail', 'Λοιπά'
UNION ALL SELECT 'panagiota.tassopoulou@cepal.gr', 'DRMU', 'Λοιπά'
UNION ALL SELECT 'sotirios.chatzigiannoulis@cepal.gr', 'FrontlineRetail', 'Λοιπά'
UNION ALL SELECT 'emanoli@cepal.gr', 'DRMU', 'Τηλεφωνική εξυπηρέτηση'
UNION ALL SELECT 'palevizou@cepal.gr', 'FrontlineRetail', 'Λοιπά'
UNION ALL SELECT 'teldmastrogianni@cepal.gr', 'Teleperformance', 'Λοιπά'
UNION ALL SELECT 'psyvkritsotaki@cepal.gr', 'Psychas', 'Λοιπά'
UNION ALL SELECT 'ekatsigaraki@cepal.gr', 'FrontlineRetail', 'Λοιπά'
UNION ALL SELECT 'atsoukala@cepal.gr', 'FrontlineRetail', 'Λοιπά'
UNION ALL SELECT 'extmkalousa@cepal.gr', 'DRMU', 'Λοιπά'
UNION ALL SELECT 'slymperopoulos@cepal.gr', 'FrontLineWholesale', 'Λοιπά'
UNION ALL SELECT 'stsantila@cepal.gr', 'DRMU', 'Τηλεφωνική εξυπηρέτηση'
UNION ALL SELECT 'tpapoulidou@cepal.gr', 'FrontlineRetail', 'Λοιπά'
UNION ALL SELECT 'vgkotzi@cepal.gr', 'FrontlineRetail', 'Λοιπά'
UNION ALL SELECT 'esalagianni@cepal.gr', 'FrontlineRetail', 'Λοιπά'
UNION ALL SELECT 'egianniou@cepal.gr', 'FrontlineRetail', 'Λοιπά'
UNION ALL SELECT 'epantazoglou@cepal.gr', 'FrontlineRetail', 'Λοιπά'
UNION ALL SELECT 'georgios.vrachas@cepal.gr', 'FrontlineRetail', 'Λοιπά'
UNION ALL SELECT 'kkiamilis@cepal.gr', 'FrontlineRetail', 'Λοιπά'
UNION ALL SELECT 'charegkavou@cepal.gr', 'Charaktiniotis', 'Λοιπά'
UNION ALL SELECT 'antonios.keramidas@cepal.gr', 'FrontlineRetail', 'Λοιπά'
UNION ALL SELECT 'atsiouri@cepal.gr', 'FrontlineRetail', 'Λοιπά'
UNION ALL SELECT 'ptselondres@cepal.gr', 'FrontLineWholesale', 'Λοιπά'
UNION ALL SELECT 'dchatzakis@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'gravmefremidou@cepal.gr', 'Marakakis', 'Λοιπά'
UNION ALL SELECT 'gdimaraki@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'empakaouka@cepal.gr', 'FrontlineRetail', 'Λοιπά'
UNION ALL SELECT 'astathakopoulou@cepal.gr', 'DRMU', 'Τηλεφωνική εξυπηρέτηση'
UNION ALL SELECT 'gkokkosis@cepal.gr', 'FrontlineRetail', 'Λοιπά'
UNION ALL SELECT 'mkremyda@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'empouzala@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'marina.effraimidou@cepal.gr', 'FrontlineRetail', 'Λοιπά'
UNION ALL SELECT 'empalai@cepal.gr', 'FrontlineRetail', 'Λοιπά'
UNION ALL SELECT 'epanagiotopoulos@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'ckolofotias@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'stavmrizoglou@cepal.gr', 'Stavropoulou', 'Λοιπά'
UNION ALL SELECT 'foteini.delli@cepal.gr', 'DRMU', 'Customer Service E - Mail'
UNION ALL SELECT 'konstantinos.anagnostou@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'okakisi@cepal.gr', 'FrontlineRetail', 'Λοιπά'
UNION ALL SELECT 'mstankoudi@cepal.gr', 'FrontlineRetail', 'Λοιπά'
UNION ALL SELECT 'ekonstantinidi@cepal.gr', 'FrontlineRetail', 'Λοιπά'
UNION ALL SELECT 'theodoros.baras@cepal.gr', 'DRMU', 'Λοιπά'
UNION ALL SELECT 'epapadopoulou@cepal.gr', 'FrontlineRetail', 'Λοιπά'
UNION ALL SELECT 'vpanagopoulos@cepal.gr', 'DRMU', 'Τηλεφωνική εξυπηρέτηση'
UNION ALL SELECT 'konstantinos.apostolopoulos@cepal.gr', 'FrontLineWHL', 'Λοιπά'
UNION ALL SELECT 'dstamatelatou@cepal.gr', 'FrontlineRetail', 'Λοιπά'
UNION ALL SELECT 'gtzalakostas@cepal.gr', 'DRMU', 'Customer Service E - Mail'
UNION ALL SELECT 'extkkolicheva@cepal.gr', 'DRMU', 'Customer Service E - Mail'
UNION ALL SELECT 'extevika@cepal.gr', 'DRMU', 'Customer Service E - Mail'
UNION ALL SELECT 'extageorgiou@cepal.gr', 'DRMU', 'Customer Service E - Mail'
UNION ALL SELECT 'ioannis.goulakos@cepal.gr', 'FrontLineWholesale', 'Λοιπά'
UNION ALL SELECT 'extnkladis@cepal.gr', 'DRMU', 'Customer Service E - Mail'
UNION ALL SELECT 'extpbouranakou@cepal.gr', 'DRMU', 'Customer Service E - Mail'
UNION ALL SELECT 'mkoviou@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'niki.stathopoulou@cepal.gr', 'DRMU', 'Λοιπά'
UNION ALL SELECT 'impatavani@cepal.gr', 'FrontLineWholesale', 'Λοιπά'
UNION ALL SELECT 'vrigopoulos@cepal.gr', 'FrontlineRetail', 'Λοιπά'
UNION ALL SELECT 'georgios.kostakis@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'extekanaki@cepal.gr', 'DRMU', 'Customer Service E - Mail'
UNION ALL SELECT 'dgiatroudaki@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'extvpatsos@cepal.gr', 'DRMU', 'Customer Service E - Mail'
UNION ALL SELECT 'konstantinos.charos@cepal.gr', 'FrontLineWHL', 'Λοιπά'
UNION ALL SELECT 'zafeiria.rappou@cepal.gr', 'FrontlineRetail', 'Λοιπά'
UNION ALL SELECT 'extapagona@cepal.gr', 'DRMU', 'Customer Service E - Mail'
UNION ALL SELECT 'extvalexopoulou@cepal.gr', 'DRMU', 'Customer Service E - Mail'
UNION ALL SELECT 'boulerosn@cepal.gr', 'FrontLineWholesale', 'Λοιπά'
UNION ALL SELECT 'eiordanidou@cepal.gr', 'DisputeTeam', 'Λοιπά'
UNION ALL SELECT 'extcchronopoulou@cepal.gr', 'DRMU', 'Customer Service E - Mail'
UNION ALL SELECT 'extmkalousa@cepal.gr', 'DisputeTeam', 'Λοιπά'
UNION ALL SELECT 'astavrianoudaki@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'extegeorgousi@cepal.gr', 'DRMU', 'Customer Service E - Mail'
UNION ALL SELECT 'sotirios.michalopoulos@cepal.gr', 'FrontLineWHL', 'Λοιπά'
UNION ALL SELECT 'theodoros.baras@cepal.gr', 'DisputeTeam', 'Λοιπά'
UNION ALL SELECT 'fstamatelatou@cepal.gr', 'DisputeTeam', 'Λοιπά'
UNION ALL SELECT 'skontou@cepal.gr', 'FrontlineRetail', 'Λοιπά'
UNION ALL SELECT 'nkatsimpiri@cepal.gr', 'FrontlineRetail', 'Λοιπά'
UNION ALL SELECT 'alexandra.gotsi@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'extamakraki@cepal.gr', 'DRMU', 'Customer Service E - Mail'
UNION ALL SELECT 'mpanteli@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'nikolaos.gkioulmpaxiotis@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'angeliki.vassalou@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'emegge@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'emastrogiannis@cepal.gr', 'FrontlineRetail', 'Λοιπά'
UNION ALL SELECT 'boulerosn@cepal.gr', 'FrontLineWHL', 'Λοιπά'
UNION ALL SELECT 'gantoniou@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'extpalexopoulou@cepal.gr', 'DRMU', 'Customer Service E - Mail'
UNION ALL SELECT 'mprekatsounaki@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'aantoniou@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'memmanouil@cepal.gr', 'FrontlineRetail', 'Λοιπά'
UNION ALL SELECT 'ioanna.liotou@cepal.gr', 'FrontLineWholesale', 'Λοιπά'
UNION ALL SELECT 'pkoutromanos@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'extacharmpas@cepal.gr', 'DRMU', 'Customer Service E - Mail'
UNION ALL SELECT 'gsiskou@cepal.gr', 'FrontlineRetail', 'Λοιπά'
UNION ALL SELECT 'extgpepkolai@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'athina.kritsivelou@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'sotsemperopoulos@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'nanagnostopoulos@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'mvosika@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'eleni.grammenou@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'alexandra.tasi@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'kspyrou@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'extizaverdinos@cepal.gr', 'null', 'Customer Service E - Mail'
UNION ALL SELECT 'exteavgeri@cepal.gr', 'null', 'Customer Service E - Mail'
UNION ALL SELECT 'charestavropoulou@cepal.gr', 'Charaktiniotis', 'Λοιπά'
UNION ALL SELECT 'alamprou@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'extepapaioannou@cepal.gr', 'null', 'Customer Service E - Mail'
UNION ALL SELECT 'extmkarampatsou@cepal.gr', 'DRMU', 'Customer Service E - Mail'
UNION ALL SELECT 'charikleia.georganta@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'vasiliki.skandali@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'eanastasiadi@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'extpmanarioti@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'extcdeligiannidou@cepal.gr', 'DRMU', 'Customer Service E - Mail'
UNION ALL SELECT 'afroditi.georgikou@cepal.gr', 'DRMU', 'Λοιπά'
UNION ALL SELECT 'dchrysou@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'sotirios.michalopoulos@cepal.gr', 'FrontLineWholesale', 'Λοιπά'
UNION ALL SELECT 'mpsimadi@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'ealexakis@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'ckontochristos@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'nikolaos.kapnisis@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'stamatia.petsi@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'eleni.nimorakiotaki@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'nikolaos.petsiavas@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'stzanis@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'eprompona@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'konstantinos.xystras@cepal.gr', 'null', 'Λοιπά'
UNION ALL SELECT 'vkardara@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'konstantinos.xystras@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'dkevga@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'kkaradima@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'kbakolas@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'garyfallia.moyze@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'panagiotis.varlamis@cepal.gr', 'FrontLineWholesale', 'Λοιπά'
UNION ALL SELECT 'pligaki@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'extmdelikoura@cepal.gr', 'DRMU', 'Customer Service E - Mail'
UNION ALL SELECT 'extdkoutonia@cepal.gr', 'DRMU', 'Customer Service E - Mail'
UNION ALL SELECT 'aikaterini.chatzimichaloglou@cepal.gr', 'FrontLineWholesale', 'Λοιπά'
UNION ALL SELECT 'aristeidis.bougioukloglou@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'omicha@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'extvvelikadi@cepal.gr', 'DRMU', 'Customer Service E - Mail'
UNION ALL SELECT 'niki.stathopoulou@cepal.gr', 'DisputeTeam', 'Λοιπά'
UNION ALL SELECT 'gliotsou@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'extitsochalis@cepal.gr', 'DRMU', 'Customer Service E - Mail'
UNION ALL SELECT 'bamstylianos', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'adamantios.andrikou@cepal.gr', 'FrontLineWholesale', 'Λοιπά'
UNION ALL SELECT 'dimitrios.papadakis@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'eleni.panagiotakopoulou@cepal.gr', 'FrontLineWholesale', 'Λοιπά'
UNION ALL SELECT 'cprassas@cepal.gr', 'MiddleOffice', 'Λοιπά'
UNION ALL SELECT 'andreas.manianis@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'styliani.chaloulakou@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'epontikakis@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'mkoulouri@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'chalexopoulou@cepal.gr', 'FrontlineRetail', 'Λοιπά'
UNION ALL SELECT 'extskiosi@cepal.gr', 'DRMU', 'Customer Service E - Mail'
UNION ALL SELECT 'mantonakou@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'dsinani@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'kchiraki@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'eevangelidis@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'skouliouras@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'extepapaioannou@cepal.gr', 'DRMU', 'Customer Service E - Mail'
UNION ALL SELECT 'akaratarakis@cepal.gr', 'FrontLineWholesale', 'Λοιπά'
UNION ALL SELECT 'masvou@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'kkazakou@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'kkatsa@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'extmnikolopoulou@cepal.gr', 'DRMU', 'Customer Service E - Mail'
UNION ALL SELECT 'vasileios.panagiotopoulos@cepal.gr', 'FrontLineWholesale', 'Λοιπά'
UNION ALL SELECT 'extpmoutopoulou@cepal.gr', 'DRMU', 'Customer Service E - Mail'
UNION ALL SELECT 'extkbeka@cepal.gr', 'DRMU', 'Customer Service E - Mail'
UNION ALL SELECT 'mspyropoulou@cepal.gr', 'CreditOperations', 'Λοιπά'
UNION ALL SELECT 'ismini.apostolopoulou@cepal.gr', 'FrontLineWholesale', 'Λοιπά'
UNION ALL SELECT 'extamakrogamvrakis@cepal.gr', 'DRMU', 'Customer Service E - Mail'
UNION ALL SELECT 'mmarkou@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'dkarmiri@cepal.gr', 'FrontlineRetail', 'Λοιπά'
UNION ALL SELECT 'stavroula.chronopoulou@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'extechatzikosta@cepal.gr', 'Sioufas', 'Λοιπά'
UNION ALL SELECT 'flora.kandioti@cepal.gr', 'FrontLineWholesale', 'Λοιπά'
UNION ALL SELECT 'extapatta@cepal.gr', 'DRMU', 'Customer Service E - Mail'
UNION ALL SELECT 'mkoutsogianni@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'georgios.balopitas@cepal.gr', 'FrontLineWholesale', 'Λοιπά'
UNION ALL SELECT 'sthomakos@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'extfgennata@cepal.gr', 'DRMU', 'Customer Service E - Mail'
UNION ALL SELECT 'extgpotiris@cepal.gr', 'DRMU', 'Customer Service E - Mail'
UNION ALL SELECT 'extamakris@cepal.gr', 'DRMU', 'Customer Service E - Mail'
UNION ALL SELECT 'nikolaos.papageorgiou@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'amourkou@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'psydgkika@cepal.gr', 'Psychas', 'Λοιπά'
UNION ALL SELECT 'frintaeleni.todri@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'ioannis.loukas@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'maria.lampa@cepal.gr', 'CreditOperations', 'Λοιπά'
UNION ALL SELECT 'extapapadopoulou@cepal.gr', 'DRMU', 'Customer Service E - Mail'
UNION ALL SELECT 'skotzageorgi@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'extdroustani@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'extigalanopoulou@cepal.gr', 'DRMU', 'Customer Service E - Mail'
UNION ALL SELECT 'extdtsantis@cepal.gr', 'DRMU', 'Customer Service E - Mail'
UNION ALL SELECT 'extefotiou@cepal.gr', 'DRMU', 'Customer Service E - Mail'
UNION ALL SELECT 'extanikolaou@cepal.gr', 'DRMU', 'Customer Service E - Mail'
UNION ALL SELECT 'nmpourna@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'extmtzikopoulou@cepal.gr', 'DRMU', 'Customer Service E - Mail'
UNION ALL SELECT 'vasileios.panagiotakopoulos@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'mariamarika.kostopoulou@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'konstantinos.apostolopoulos@cepal.gr', 'FrontLineWholesale', 'Λοιπά'
UNION ALL SELECT 'okavarinou@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'georgia.logariastaki@cepal.gr', 'FrontlineRetail', 'Λοιπά'
UNION ALL SELECT 'nkoi@cepal.gr', 'FrontlineRetail', 'Λοιπά'
UNION ALL SELECT 'ppantsidis@cepal.gr', 'FrontLineWholesale', 'Λοιπά'
UNION ALL SELECT 'extmptochou@cepal.gr', 'DRMU', 'Customer Service E - Mail'
UNION ALL SELECT 'konstantinos.charos@cepal.gr', 'FrontLineWholesale', 'Λοιπά'
UNION ALL SELECT 'maria.panagopoulou@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'esmargianakis@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'emmanouil.anastasakis@cepal.gr', 'FrontLineWholesale', 'Λοιπά'
UNION ALL SELECT 'vmpesios@cepal.gr', 'FrontLineWholesale', 'Λοιπά'
UNION ALL SELECT 'gdimaraki@cepal.gr', 'FrontLineRTL', 'Λοιπά'
UNION ALL SELECT 'eparousi@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'pkatsarou@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'christos.malamatinas@cepal.gr', 'FrontLineWholesale', 'Λοιπά'
UNION ALL SELECT 'extesoupidou@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'aioannidou@cepal.gr', 'DisputeTeam', 'Λοιπά'
UNION ALL SELECT 'achrysaiti@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'mkeskou@cepal.gr', 'FrontlineRetail', 'Λοιπά'
UNION ALL SELECT 'christos.malamatinas@cepal.gr', 'FrontLineWHL', 'Λοιπά'
UNION ALL SELECT 'sfilippopoulos@cepal.gr', 'FrontLineWholesale', 'Λοιπά'
UNION ALL SELECT 'ininos@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'ftsotsou@cepal.gr', 'FrontlineRetail', 'Λοιπά'
UNION ALL SELECT 'melfandrikopoulou@cepal.gr', 'DRMU', 'Λοιπά'
UNION ALL SELECT 'agkogkakis@cepal.gr', 'DRMU', 'Customer Service E - Mail'
UNION ALL SELECT 'sgerontaki@cepal.gr', 'FrontlineRetail', 'Λοιπά'
UNION ALL SELECT 'agkogkakis', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'amourkou', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'evatsironissarantopo', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'marnikolopoulou', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'gsallaris', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'thepetropoulou', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'ddelimpalta', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'thebaras', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'nntoulia', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'eprompona', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'mstankoudi', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'casimakopoulos', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'athgkana', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'nmpourna', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'sotsemperopoulos', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'extadimitriou', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'ekremmyda', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'marpanagopoulou', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'aioannidou', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'floukas', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'okavarinou', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'ioaloukas', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'vasskandali', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'zoilazou', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'konchrysochoidis', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'kkazakou', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'extmalimpate', 'CEPAL', 'Customer Service E - Mail'
UNION ALL SELECT 'pantassopoulou', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'lamtasioula', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'chageorganta', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'tpapapolyzou', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'cprassas', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'gtkkarvela', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'mareffraimidou', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'alamprou', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'evakokodroulis', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'omargariti', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'elipapaleonida', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'skotzageorgi', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'epanagiotopoulos', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'sotmichalopoulos', 'CEPAL', 'Λοιπά'
UNION ALL SELECT 'extcpanagodimitropoulou@cepal.gr', 'DRMU', 'Customer Service E - Mail'
UNION ALL SELECT 'antonios.makryniotis@cepal.gr', 'FrontLineWholesale', 'Λοιπά'
UNION ALL SELECT 'christos.perdikis@cepal.gr', 'FrontlineRetail', 'Λοιπά'
UNION ALL SELECT 'panagiotaev.vangelatou@cepal.gr', 'FrontlineRetail', 'Λοιπά'
UNION ALL SELECT 'izisaki@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'viliadou@cepal.gr', 'FrontLineWholesale', 'Λοιπά'
UNION ALL SELECT 'ekokoli@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'ppantsidis@cepal.gr', 'FrontLineWHL', 'Λοιπά'
UNION ALL SELECT 'vmpesios@cepal.gr', 'FrontLineWHL', 'Λοιπά'
UNION ALL SELECT 'flora.kandioti@cepal.gr', 'FrontLineWHL', 'Λοιπά'
UNION ALL SELECT 'lamprini.tasioula@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'extknasioudis@cepal.gr', 'Charaktiniotis', 'Λοιπά'
UNION ALL SELECT 'georgios.katalagargianos@cepal.gr', 'FrontLineWholesale', 'Λοιπά'
UNION ALL SELECT 'skyrkos@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'ekremmyda@cepal.gr', 'FrontLineRetail', 'Λοιπά'
UNION ALL SELECT 'mchatzisaroglou - ΧΑΤΖΗΣΑΡΟΓΛΟΥ ΜΑΡΙΑ', 'mchatzisaroglou', 'Τηλεφωνική εξυπηρέτηση'
UNION ALL SELECT 'msotiriadou - ΣΩΤΗΡΙΑΔΟΥ ΜΑΡΙΑ', 'msotiriadou', 'Τηλεφωνική εξυπηρέτηση'
UNION ALL SELECT 'atsekoura - ΤΣΕΚΟΥΡΑ ΑΓΓΕΛΙΚΗ', 'atsekoura', 'Τηλεφωνική εξυπηρέτηση'
UNION ALL SELECT 'extekanaki - ΚΑΝΑΚΗ ΕΛΕΥΘΕΡΙΑ', 'extekanaki', 'Customer Service E - Mail'
UNION ALL SELECT 'astathakopoulou - ΣΤΑΘΑΚΟΠΟΥΛΟΥ ΑΘΗΝΑ', 'astathakopoulou', 'Τηλεφωνική εξυπηρέτηση'
UNION ALL SELECT 'vpanagopoulos - ΠΑΝΑΓΟΠΟΥΛΟΣ ΒΑΣΙΛΕΙΟΣ', 'vpanagopoulos', 'Τηλεφωνική εξυπηρέτηση'
UNION ALL SELECT 'emanoli - ΜΑΝΩΛΗ ΕΛΕΝΗ', 'emanoli', 'Τηλεφωνική εξυπηρέτηση'
UNION ALL SELECT 'extdkoutonia - ΚΟΥΤΟΝΙΑ ΔΙΟΝΥΣΙΑ', 'extdkoutonia', 'Customer Service E - Mail'
UNION ALL SELECT 'extpbouranakou - ΜΠΟΥΡΑΝΑΚΟΥ ΠΑΝΑΓΙΩΤΑ', 'extpbouranakou', 'Customer Service E - Mail'
UNION ALL SELECT 'stsantila - ΤΣΑΝΤΗΛΑ ΣΤΑΥΡΟΥΛΑ', 'stsantila', 'Τηλεφωνική εξυπηρέτηση'
UNION ALL SELECT 'extkbeka - ΜΠΕΚΑ ΚΩΝΤΑΝΤΙΝΑ', 'extkbeka', 'Customer Service E - Mail'
UNION ALL SELECT 'vpanagopoulos - ΠΑΝΑΓΟΠΟΥΛΟΣ ΒΑΣΙΛΕΙΟΣ', 'system', 'Τηλεφωνική εξυπηρέτηση'
UNION ALL SELECT 'emanoli - ΜΑΝΩΛΗ ΕΛΕΝΗ', 'system', 'Τηλεφωνική εξυπηρέτηση'
UNION ALL SELECT 'IVR - IVR IVR', 'IVR', 'Τηλεφωνική εξυπηρέτηση'
UNION ALL SELECT 'IVR - IVR IVR', 'system', 'Τηλεφωνική εξυπηρέτηση'
UNION ALL SELECT 'extfgennata - ΓΕΝΝΑΤΑ ΦΑΝΗ', 'extfgennata', 'Customer Service E - Mail'
UNION ALL SELECT 'extamakraki - ΜΑΚΡΑΚΗ ΑΛΕΞΑΝΔΡΑ', 'extamakraki', 'Customer Service E - Mail'
UNION ALL SELECT 'extapatta - Παττα Αναστασια', 'extapatta', 'Customer Service E - Mail'
UNION ALL SELECT 'extcchronopoulou - ΧΡΟΝΟΠΟΥΛΟΥ ΧΡΙΣΤΙΝΑ', 'extcchronopoulou', 'Customer Service E - Mail'
UNION ALL SELECT 'extkkolicheva - KOLICHEVA KATERINA', 'extkkolicheva', 'Customer Service E - Mail'
UNION ALL SELECT 'x94817 - ΠΛΑΤΑΝΑ ΕΙΡΗΝΗ', 'x94817', 'Λοιπά'
UNION ALL SELECT 'x94811 - ΓΚΡΙΝΙΑ ΙΩΑΝΝΑ', 'x94811', 'Λοιπά'
UNION ALL SELECT 'extvpatsos - ΠΑΤΣΟΣ ΒΑΣΙΛΕΙΟΣ', 'extvpatsos', 'Customer Service E - Mail'
UNION ALL SELECT 'extmnikolopoulou - ΝΙΚΟΛΟΠΟΥΛΟΥ ΜΑΡΙΝΑ', 'extmnikolopoulou', 'Customer Service E - Mail'
UNION ALL SELECT 'extepapaioannou - ΠΑΠΑΙΩΑΝΝΟΥ ΕΛΕΝΗ', 'extepapaioannou', 'Customer Service E - Mail'
UNION ALL SELECT 'extmtzikopoulou - ΤΖΙΚΟΠΟΥΛΟΥ ΜΑΡΙΑ ΠΟΛΥΞΕΝΗ', 'extmtzikopoulou', 'Customer Service E - Mail'
UNION ALL SELECT 'extpmoutopoulou - ΜΟΥΤΟΠΟΥΛΟΥ ΠΑΡΑΣΚΕΥΗ', 'extpmoutopoulou', 'Customer Service E - Mail'
UNION ALL SELECT 'x94649 - ΤΑΣΣΟΠΟΥΛΟΥ ΠΑΝΑΓΙΩΤΑ', 'x94649', 'Λοιπά'
UNION ALL SELECT 'extacharmpas - ΧΑΡΜΠΑΣ ΑΝΑΣΤΑΣΙΟΣ', 'extacharmpas', 'Customer Service E - Mail'
UNION ALL SELECT 'fstamatelatou - ΣΤΑΜΑΤΕΛΑΤΟΥ ΦΩΤΕΙΝΗ', 'fstamatelatou', 'Λοιπά'
UNION ALL SELECT 'extdtsantis - ΤΣΑΝΤΗΣ ΔΗΜΟΣΘΕΝΗΣ', 'extdtsantis', 'Customer Service E - Mail'
UNION ALL SELECT 'extitsochalis - Τσοχαλης Ιωαννης', 'extitsochalis', 'Customer Service E - Mail'
UNION ALL SELECT 'extgpotiris - ΠΟΤΗΡΗΣ ΓΙΩΡΓΟΣ', 'extgpotiris', 'Customer Service E - Mail'
UNION ALL SELECT 'extanikolaou - ΝΙΚΟΛΑΟΥ ΑΙΚΑΤΕΡΙΝΗ', 'extanikolaou', 'Customer Service E - Mail'
UNION ALL SELECT 'x94492 - ΜΠΑΡΑΣ ΘΕΟΔΩΡΟΣ', 'x94492', 'Λοιπά'
UNION ALL SELECT 'extapapadopoulou - ΠΑΠΑΔΟΠΟΥΛΟΥ ΑΙΚΑΤΕΡΙΝΗ', 'extapapadopoulou', 'Customer Service E - Mail'
UNION ALL SELECT 'extamakrogamvrakis - ΜΑΚΡΟΓΑΜΒΡΑΚΗΣ ΑΝΤΩΝΙΟΣ', 'extamakrogamvrakis', 'Customer Service E - Mail'
UNION ALL SELECT 'extefotiou - ΦΩΤΙΟΥ ΕΜΜΑΝΟΥΗΛ', 'extefotiou', 'Customer Service E - Mail'
UNION ALL SELECT 'extevika - ΒΙΚΑ ΕΥΑΓΓΕΛΙΑ', 'extevika', 'Customer Service E - Mail'
UNION ALL SELECT 'extmkalousa - ΚΑΛΟΥΣΑ ΜΑΡΙΑ-ΛΟΥΙΖΑ', 'extmkalousa', 'Λοιπά'
UNION ALL SELECT 'stsantila - ΤΣΑΝΤΗΛΑ ΣΤΑΥΡΟΥΛΑ', 'system', 'Τηλεφωνική εξυπηρέτηση'
UNION ALL SELECT 'extigalanopoulou - ΓΑΛΑΝΟΠΟΥΛΟΥ ΙΦΙΓΕΝΕΙΑ', 'extigalanopoulou', 'Customer Service E - Mail'
UNION ALL SELECT 'extmkarampatsou - ΜΑΡΙΛΕΝΑ ΚΑΡΑΜΠΑΤΣΟΥ', 'extmkarampatsou', 'Customer Service E - Mail'
UNION ALL SELECT 'ipalmos - ΠΑΛΜΟΣ ΙΩΑΝΝΗΣ', 'ipalmos', 'Λοιπά'
UNION ALL SELECT 'aioannidoy - ΙΩΑΝΝΙΔΟΥ ΑΝΑΣΤΑΣΙΑ', 'aioannidoy', 'Λοιπά'
UNION ALL SELECT 'eiordanidou - ΙΟΡΔΑΝΙΔΟΥ ΕΛΕΝΗ', 'eiordanidou', 'Λοιπά'
UNION ALL SELECT 'X88523 - ΤΖΑΒΕΛΛΑ ΑΘΑΝΑΣΙΑ', 'X88523', 'Τηλεφωνική εξυπηρέτηση'
UNION ALL SELECT 'emanoli - Manoli Eleni', 'emanoli', 'Τηλεφωνική εξυπηρέτηση'
UNION ALL SELECT 'mchatzisaroglou - Chatzisaroglou Maria', 'mchatzisaroglou', 'Τηλεφωνική εξυπηρέτηση'
UNION ALL SELECT 'atsekoura - Tsekoura Aggeliki', 'atsekoura', 'Τηλεφωνική εξυπηρέτηση'
UNION ALL SELECT 'vpanagopoulos - Panagopoulos Vasileios', 'vpanagopoulos', 'Τηλεφωνική εξυπηρέτηση'
UNION ALL SELECT 'stsantila - Tsantila Stavroula', 'stsantila', 'Τηλεφωνική εξυπηρέτηση'
UNION ALL SELECT 'extpbouranakou - Bouranakou Panagiota', 'extpbouranakou', 'Customer Service E - Mail'
UNION ALL SELECT 'extcchronopoulou - Chronopoulou Christina', 'extcchronopoulou', 'Customer Service E - Mail'
UNION ALL SELECT 'extitsochalis - Tsochalis Ioannis', 'extitsochalis', 'Customer Service E - Mail'
UNION ALL SELECT 'extmnikolopoulou - Nikolopoulou Marina', 'extmnikolopoulou', 'Customer Service E - Mail'
UNION ALL SELECT 'extamakraki - Makraki Alexandra', 'extamakraki', 'Customer Service E - Mail'
UNION ALL SELECT 'extmtzikopoulou - Tzikopoulou Maria-Polyxeni', 'extmtzikopoulou', 'Customer Service E - Mail'
UNION ALL SELECT 'msotiriadou - Sotiriadou Maria', 'msotiriadou', 'Τηλεφωνική εξυπηρέτηση'
UNION ALL SELECT 'extvpatsos - Patsos Vasileios', 'extvpatsos', 'Customer Service E - Mail'
UNION ALL SELECT 'astathakopoulou - Stathakopoulou Athina', 'astathakopoulou', 'Τηλεφωνική εξυπηρέτηση'
UNION ALL SELECT 'extdkoutonia - Koutonia Dionysia', 'extdkoutonia', 'Customer Service E - Mail'
UNION ALL SELECT 'ioagkrinia - Gkrinia Ioanna', 'ioagkrinia', 'Λοιπά'
UNION ALL SELECT 'extanikolaou - Nikolaou Aikaterini', 'extanikolaou', 'Customer Service E - Mail'
UNION ALL SELECT 'eirplatana - Platana Eirini', 'eirplatana', 'Λοιπά'
UNION ALL SELECT 'extigalanopoulou - Galanopoulou Ifigeneia', 'extigalanopoulou', 'Customer Service E - Mail'
UNION ALL SELECT 'emanoli - Manoli Eleni', 'system', 'Τηλεφωνική εξυπηρέτηση'
UNION ALL SELECT 'null', 'system', 'Λοιπά'
UNION ALL SELECT 'extepapaioannou - Papaioannou Eleni', 'extepapaioannou', 'Customer Service E - Mail'
UNION ALL SELECT 'extfgennata - Gennata Fani', 'extfgennata', 'Customer Service E - Mail'
UNION ALL SELECT 'extapapadopoulou - Papadopoulou  Aikaterini', 'extapapadopoulou', 'Customer Service E - Mail'
UNION ALL SELECT 'extefotiou - Fotiou Emmanouil', 'extefotiou', 'Customer Service E - Mail'
UNION ALL SELECT 'extgpotiris - Potiris Giorgos', 'extgpotiris', 'Customer Service E - Mail'
UNION ALL SELECT 'extacharmpas - Charmpas Anastasios', 'extacharmpas', 'Customer Service E - Mail'
UNION ALL SELECT 'extdtsantis - Tsantis Dimosthenis', 'extdtsantis', 'Customer Service E - Mail'
UNION ALL SELECT 'extapatta - Patta Anastasia', 'extapatta', 'Customer Service E - Mail'
UNION ALL SELECT 'extevika - Vika Evaggelia', 'extevika', 'Customer Service E - Mail'
UNION ALL SELECT 'extamakrogamvrakis - Makrogamvrakis Antonios', 'extamakrogamvrakis', 'Customer Service E - Mail'
UNION ALL SELECT 'extmkarampatsou - Karampatsou Marilena', 'extmkarampatsou', 'Customer Service E - Mail'
UNION ALL SELECT 'extkbeka - Mpeka Konstantina', 'extkbeka', 'Customer Service E - Mail'
UNION ALL SELECT 'aioannidou - Ioannidou Natasa', 'aioannidou', 'Λοιπά'
UNION ALL SELECT 'extpmoutopoulou - Moutopoulou Paraskevi', 'extpmoutopoulou', 'Customer Service E - Mail'
UNION ALL SELECT 'pantassopoulou - Tassopoulou Panagiota', 'pantassopoulou', 'Λοιπά'
UNION ALL SELECT 'extekanaki - Kanaki Eleftheria', 'extekanaki', 'Customer Service E - Mail'
UNION ALL SELECT 'extskiosi - ΚΙΟΣΗ ΣΠΥΡΙΔΟΥΛΑ', 'extskiosi', 'Customer Service E - Mail'
UNION ALL SELECT 'melepoliti - ΠΟΛΙΤΗ ΕΥΗ', 'melepoliti', 'Λοιπά'
UNION ALL SELECT 'extegeorgousi - ΓΕΩΡΓΟΥΣΗ ΕΥΦΡΟΣΥΝΗ', 'extegeorgousi', 'Customer Service E - Mail'
UNION ALL SELECT 'extmptochou - ΠΤΩΧΟΥ ΜΑΡΙΑ', 'extmptochou', 'Customer Service E - Mail'
UNION ALL SELECT 'extcpanagodimitropou - ΠΑΝΑΓΟΔΗΜΗΤΡΟΠΟΥΛΟΥ ΧΡΥΣΟΥΛΑ', 'extcpanagodimitropou', 'Customer Service E - Mail'
UNION ALL SELECT 'extdkoutonia - ΚΟΥΤΟΝΙΑ ΔΙΟΝΥΣΙΑ', 'system', 'Customer Service E - Mail'
UNION ALL SELECT 'extamakris - Makris Argyris', 'extamakris', 'Customer Service E - Mail'
UNION ALL SELECT 'extcpanagodimitropou - Panagodimitropoulou Chrysoula', 'extcpanagodimitropou', 'Customer Service E - Mail'
UNION ALL SELECT 'extmkalousa - Kalousa Maria-Louiza', 'extmkalousa', 'Λοιπά'
UNION ALL SELECT 'eiordanidou - Iordanidou Eleni', 'eiordanidou', 'Λοιπά';